# LSI và tóm tắt dựa trên không gian ngữ nghĩa

**Nội dung:** (1) mô phỏng từng bước trên văn bản nhỏ; (2) ứng dụng tóm tắt
trích xuất; (3) thử nghiệm trên dữ liệu công khai và đánh giá.

TF, DF, IDF, TF-IDF, chọn câu và ROUGE được cài đặt trực tiếp trong notebook.
Phân rã SVD dùng `numpy.linalg.svd`. Không dùng `TfidfVectorizer`, thư viện
tóm tắt hay thư viện ROUGE.

**Chạy:** mở notebook bằng Jupyter và chọn **Run → Run All Cells**.
Cài thư viện bằng `python -m pip install -r requirements.txt` khi cần.
Dữ liệu 50 bài báo đã kèm trong thư mục `data`; notebook đọc trực tiếp tệp này.

## Chuẩn bị thư viện

In [1]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 4)

# Phần 1 - Mô phỏng thuật toán trên dữ liệu nhỏ

## 1.1. Văn bản đầu vào

Mỗi câu được xem là một tài liệu nhỏ. Ma trận của một văn bản có **hàng là
từ, cột là câu**. Ví dụ dưới đây gồm hai nhóm nội dung: ngôn ngữ/văn bản
và bóng đá/đội bóng.

In [2]:
demo_text = """Máy tính xử lý ngôn ngữ.
Ngôn ngữ giúp máy tính hiểu văn bản.
LSI giúp tóm tắt văn bản.
Tóm tắt chọn câu trong văn bản.
Đội bóng thi đấu bóng đá.
Bóng đá cần đội bóng mạnh."""

display(Markdown(demo_text.replace("\n", "  \n")))

Máy tính xử lý ngôn ngữ.  
Ngôn ngữ giúp máy tính hiểu văn bản.  
LSI giúp tóm tắt văn bản.  
Tóm tắt chọn câu trong văn bản.  
Đội bóng thi đấu bóng đá.  
Bóng đá cần đội bóng mạnh.

## 1.2. Tách câu và tiền xử lý

Chuẩn hóa Unicode, chuyển chữ thường, lấy token chữ/số và loại từ dừng.
Danh sách từ dừng nhỏ được ghi rõ trong code. Với tiếng Việt, token ở đây
là âm tiết, chưa ghép các từ nhiều âm tiết như “máy tính”.

In [3]:
STOP_WORDS = set("""
a an the and or but is are was were be been being to of in on at for from
with by as that this these those it its he she they we you i his her their
our not do does did have has had will would can could should may might
và là của có được một những các cho với đã đang sẽ này đó ở để thì mà
""".split())


def split_sentences(text):
    text = unicodedata.normalize("NFC", text)
    parts = re.split(r'(?<=[.!?])\s+|(?<=[.!?]["”’])\s+|\n+', text)
    return list(filter(None, map(str.strip, parts)))


def tokenize(text):
    text = unicodedata.normalize("NFC", text).lower()
    return re.findall(r"\b\w+\b", text)


def preprocess(sentence):
    return list(filter(lambda word: word not in STOP_WORDS, tokenize(sentence)))


demo_sentences = split_sentences(demo_text)
demo_tokens = list(map(preprocess, demo_sentences))
sentence_names = [f"S{index + 1}" for index in range(len(demo_sentences))]
display(pd.DataFrame({"Câu": demo_sentences, "Token sau tiền xử lý": demo_tokens},
                     index=sentence_names))

,Câu,Token sau tiền xử lý
S1,Máy tính xử lý ngôn ngữ.,"[máy, tính, xử, lý, ngôn, ngữ]"
S2,Ngôn ngữ giúp máy tính hiểu văn bản.,"[ngôn, ngữ, giúp, máy, tính, hiểu, văn, bản]"
S3,LSI giúp tóm tắt văn bản.,"[lsi, giúp, tóm, tắt, văn, bản]"
S4,Tóm tắt chọn câu trong văn bản.,"[tóm, tắt, chọn, câu, trong, văn, bản]"
S5,Đội bóng thi đấu bóng đá.,"[đội, bóng, thi, đấu, bóng, đá]"
S6,Bóng đá cần đội bóng mạnh.,"[bóng, đá, cần, đội, bóng, mạnh]"


## 1.3. Tự cài đặt TF-IDF

Với $N$ câu và từ $t$:

$$TF(t,s_j)=\text{số lần từ }t\text{ xuất hiện trong câu }s_j$$
$$DF(t)=\text{số câu chứa từ }t$$
$$IDF(t)=\ln\frac{N+1}{DF(t)+1}+1$$
$$W_{t,j}=TF(t,s_j)\,IDF(t)$$
$$A_{t,j}=\frac{W_{t,j}}{\sqrt{\sum_t W_{t,j}^{2}}}$$

Cộng 1 để làm trơn IDF. Chuẩn hóa L2 giúp giảm ảnh hưởng của độ dài câu.
Cột không có token hợp lệ được giữ bằng 0. Văn bản đầu vào cần có ít nhất
một token hợp lệ sau tiền xử lý. Các giá trị trung gian được trả về để quan sát.

In [4]:
def build_tfidf(sentences):
    tokenized_sentences = list(map(preprocess, sentences))
    vocabulary = sorted(set(word for tokens in tokenized_sentences for word in tokens))

    counts = list(map(Counter, tokenized_sentences))
    tf = np.array([[count[word] for count in counts] for word in vocabulary], dtype=float)
    df = np.count_nonzero(tf, axis=1)
    idf = np.log((len(sentences) + 1) / (df + 1)) + 1
    weights = tf * idf[:, np.newaxis]
    column_norms = np.linalg.norm(weights, axis=0)
    matrix = weights / np.maximum(column_norms, np.finfo(float).eps)

    return {"vocabulary": vocabulary, "tf": tf, "df": df, "idf": idf,
            "weights": weights, "matrix": matrix}


demo_tfidf = build_tfidf(demo_sentences)
display(Markdown("**TF: số lần xuất hiện của từng từ trong từng câu**"))
display(pd.DataFrame(demo_tfidf["tf"], index=demo_tfidf["vocabulary"], columns=sentence_names))
display(Markdown("**DF và IDF**"))
display(pd.DataFrame({"DF": demo_tfidf["df"], "IDF": demo_tfidf["idf"]},
                     index=demo_tfidf["vocabulary"]))

**TF: số lần xuất hiện của từng từ trong từng câu**

,S1,S2,S3,S4,S5,S6
bóng,0.0,0.0,0.0,0.0,2.0,2.0
bản,0.0,1.0,1.0,1.0,0.0,0.0
chọn,0.0,0.0,0.0,1.0,0.0,0.0
câu,0.0,0.0,0.0,1.0,0.0,0.0
cần,0.0,0.0,0.0,0.0,0.0,1.0
giúp,0.0,1.0,1.0,0.0,0.0,0.0
hiểu,0.0,1.0,0.0,0.0,0.0,0.0
lsi,0.0,0.0,1.0,0.0,0.0,0.0
lý,1.0,0.0,0.0,0.0,0.0,0.0
máy,1.0,1.0,0.0,0.0,0.0,0.0


**DF và IDF**

,DF,IDF
bóng,2,1.8473
bản,3,1.5596
chọn,1,2.2528
câu,1,2.2528
cần,1,2.2528
giúp,2,1.8473
hiểu,1,2.2528
lsi,1,2.2528
lý,1,2.2528
máy,2,1.8473


In [5]:
display(Markdown("**TF × IDF trước chuẩn hóa**"))
display(pd.DataFrame(demo_tfidf["weights"], index=demo_tfidf["vocabulary"], columns=sentence_names))
display(Markdown("**Ma trận A: TF-IDF sau chuẩn hóa L2**"))
display(pd.DataFrame(demo_tfidf["matrix"], index=demo_tfidf["vocabulary"], columns=sentence_names))

**TF × IDF trước chuẩn hóa**

,S1,S2,S3,S4,S5,S6
bóng,0.0000,0.0000,0.0000,0.0000,3.6946,3.6946
bản,0.0000,1.5596,1.5596,1.5596,0.0000,0.0000
chọn,0.0000,0.0000,0.0000,2.2528,0.0000,0.0000
câu,0.0000,0.0000,0.0000,2.2528,0.0000,0.0000
cần,0.0000,0.0000,0.0000,0.0000,0.0000,2.2528
giúp,0.0000,1.8473,1.8473,0.0000,0.0000,0.0000
hiểu,0.0000,2.2528,0.0000,0.0000,0.0000,0.0000
lsi,0.0000,0.0000,2.2528,0.0000,0.0000,0.0000
lý,2.2528,0.0000,0.0000,0.0000,0.0000,0.0000
máy,1.8473,1.8473,0.0000,0.0000,0.0000,0.0000


**Ma trận A: TF-IDF sau chuẩn hóa L2**

,S1,S2,S3,S4,S5,S6
bóng,0.0000,0.0000,0.0000,0.0000,0.6676,0.6676
bản,0.0000,0.3001,0.3472,0.3006,0.0000,0.0000
chọn,0.0000,0.0000,0.0000,0.4342,0.0000,0.0000
câu,0.0000,0.0000,0.0000,0.4342,0.0000,0.0000
cần,0.0000,0.0000,0.0000,0.0000,0.0000,0.4071
giúp,0.0000,0.3555,0.4112,0.0000,0.0000,0.0000
hiểu,0.0000,0.4335,0.0000,0.0000,0.0000,0.0000
lsi,0.0000,0.0000,0.5015,0.0000,0.0000,0.0000
lý,0.4618,0.0000,0.0000,0.0000,0.0000,0.0000
máy,0.3787,0.3555,0.0000,0.0000,0.0000,0.0000


## 1.4. SVD và không gian ngữ nghĩa

$$A=U\Sigma V^\top \approx U_k\Sigma_kV_k^\top$$

Giữ $k$ giá trị kỳ dị lớn nhất. Tọa độ ngữ nghĩa của các câu là:

$$Z=\Sigma_kV_k^\top$$

Cột $j$ của $Z$ biểu diễn câu $j$ trong không gian $k$ chiều. Các chiều
phản ánh mẫu đồng xuất hiện của từ, không phải nhãn chủ đề đã biết trước.
SVD dùng phép phân rã số học của NumPy; bước rút gọn và chấm điểm được
cài đặt trực tiếp bên dưới.

In [6]:
demo_k = 2
demo_u, demo_singular_values, demo_vt = np.linalg.svd(demo_tfidf["matrix"], full_matrices=False)
demo_k = min(demo_k, len(demo_singular_values))
dimension_names = [f"Chiều {index + 1}" for index in range(len(demo_singular_values))]

display(Markdown("**U: quan hệ giữa từ và chiều ngữ nghĩa**"))
display(pd.DataFrame(demo_u, index=demo_tfidf["vocabulary"], columns=dimension_names))
display(Markdown("**Các giá trị kỳ dị**"))
display(pd.DataFrame({"sigma": demo_singular_values}, index=dimension_names))
display(Markdown("**Vᵀ: quan hệ giữa chiều ngữ nghĩa và câu**"))
display(pd.DataFrame(demo_vt, index=dimension_names, columns=sentence_names))

**U: quan hệ giữa từ và chiều ngữ nghĩa**

,Chiều 1,Chiều 2,Chiều 3,Chiều 4,Chiều 5,Chiều 6
bóng,-6.4787e-17,7.3092e-01,-4.9249e-17,6.5926e-17,5.1010e-17,1.2916e-16
bản,-3.7240e-01,-5.7435e-17,-1.7091e-01,-9.0078e-02,1.3883e-01,-8.5395e-18
chọn,-1.4812e-01,-7.2584e-17,-1.9415e-01,4.1021e-01,1.5511e-01,-1.0378e-17
câu,-1.4812e-01,-8.2189e-17,-1.9415e-01,4.1021e-01,1.5511e-01,-1.1664e-17
cần,-1.8973e-17,2.2284e-01,-2.1974e-17,1.6384e-17,-5.8805e-18,-5.0000e-01
giúp,-3.1964e-01,4.6155e-18,-4.3233e-02,-4.4307e-01,3.7243e-02,3.4540e-18
hiểu,-1.8976e-01,-1.1534e-17,1.3695e-01,-1.8831e-01,4.8107e-01,-9.7506e-19
lsi,-2.0003e-01,1.7842e-17,-1.8967e-01,-3.5201e-01,-4.3565e-01,3.6629e-18
lý,-1.3302e-01,5.7375e-17,2.6462e-01,2.3701e-01,-4.1850e-01,2.6687e-17
máy,-2.6469e-01,4.3403e-17,3.2929e-01,3.9935e-02,5.1306e-02,1.9132e-17


**Các giá trị kỳ dị**

,sigma
Chiều 1,1.3484
Chiều 2,1.2917
Chiều 3,1.1388
Chiều 4,0.7378
Chiều 5,0.5837
Chiều 6,0.5757


**Vᵀ: quan hệ giữa chiều ngữ nghĩa và câu**

,S1,S2,S3,S4,S5,S6
Chiều 1,-0.3884,-0.5902,-5.3782e-01,-4.5993e-01,-6.7376e-17,-6.2845e-17
Chiều 2,-0.0000,-0.0000,-6.0788e-17,-1.2912e-16,7.0711e-01,7.0711e-01
Chiều 3,0.6526,0.3597,-4.3068e-01,-5.0915e-01,-6.8527e-17,-6.1472e-17
Chiều 4,0.3787,-0.3205,-5.1784e-01,6.9696e-01,5.3059e-17,2.9693e-17
Chiều 5,-0.5290,0.6477,-5.0706e-01,2.0851e-01,2.9178e-18,-8.4321e-18
Chiều 6,-0.0000,-0.0000,8.7538e-18,-1.7202e-17,7.0711e-01,-7.0711e-01


In [7]:
demo_coordinates = demo_singular_values[:demo_k, np.newaxis] * demo_vt[:demo_k]
demo_approximation = demo_u[:, :demo_k] @ demo_coordinates

display(Markdown(f"**Z: tọa độ câu trong {demo_k} chiều ngữ nghĩa**"))
display(pd.DataFrame(demo_coordinates, index=dimension_names[:demo_k], columns=sentence_names))
display(Markdown("**Aₖ: ma trận xấp xỉ sau rút gọn**"))
display(pd.DataFrame(demo_approximation, index=demo_tfidf["vocabulary"], columns=sentence_names))

**Z: tọa độ câu trong 2 chiều ngữ nghĩa**

,S1,S2,S3,S4,S5,S6
Chiều 1,-0.5238,-0.7958,-7.2518e-01,-6.2016e-01,-9.0849e-17,-8.4739e-17
Chiều 2,-0.0000,-0.0000,-7.8522e-17,-1.6679e-16,9.1339e-01,9.1339e-01


**Aₖ: ma trận xấp xỉ sau rút gọn**

,S1,S2,S3,S4,S5,S6
bóng,3.3933e-17,5.1559e-17,-1.0411e-17,-8.1734e-17,6.6762e-01,6.6762e-01
bản,1.9505e-01,2.9636e-01,2.7006e-01,2.3095e-01,-1.8629e-17,-2.0904e-17
chọn,7.7577e-02,1.1787e-01,1.0741e-01,9.1856e-02,-5.2842e-17,-5.3747e-17
câu,7.7577e-02,1.1787e-01,1.0741e-01,9.1856e-02,-6.1615e-17,-6.2520e-17
cần,9.9372e-18,1.5099e-17,-3.7388e-18,-2.5401e-17,2.0354e-01,2.0354e-01
giúp,1.6741e-01,2.5437e-01,2.3179e-01,1.9823e-01,3.3254e-17,3.1301e-17
hiểu,9.9388e-02,1.5101e-01,1.3761e-01,1.1768e-01,6.7043e-18,5.5450e-18
lsi,1.0477e-01,1.5919e-01,1.4506e-01,1.2405e-01,3.4469e-17,3.3247e-17
lý,6.9673e-02,1.0586e-01,9.6467e-02,8.2497e-02,6.4491e-17,6.3679e-17
máy,1.3863e-01,2.1064e-01,1.9195e-01,1.6415e-01,6.3690e-17,6.2073e-17


## 1.5. Chấm điểm và chọn câu

Sử dụng độ lớn tọa độ trong không gian ngữ nghĩa:

$$score(s_j)=\sqrt{\sum_{i=1}^{k}(\sigma_iV^\top_{i,j})^2}$$

Đây là quy tắc chọn câu của bài làm, không phải công thức duy nhất cho
tóm tắt LSI. Chọn các câu có điểm cao nhất, sau đó trả về theo thứ tự gốc.
Điểm được làm tròn 12 chữ số khi xếp hạng để tránh sai số SVD rất nhỏ
làm thay đổi thứ tự của các câu đồng điểm.

In [8]:
def semantic_scores(singular_values, vt, k):
    k = min(k, len(singular_values))
    coordinates = singular_values[:k, np.newaxis] * vt[:k]
    return np.linalg.norm(coordinates, axis=0)


def select_sentences(sentences, scores, sentence_count):
    ranking = np.argsort(-np.round(scores, 12), kind="stable")
    selected_indices = np.sort(ranking[:sentence_count])
    return selected_indices, " ".join(sentences[index] for index in selected_indices)


demo_sentence_count = 2
demo_scores = semantic_scores(demo_singular_values, demo_vt, demo_k)
demo_selected, demo_summary = select_sentences(demo_sentences, demo_scores, demo_sentence_count)
demo_ranking = pd.DataFrame({"Câu": demo_sentences, "Điểm LSI": demo_scores,
                             "Được chọn": np.isin(np.arange(len(demo_sentences)), demo_selected)},
                            index=sentence_names)
display(demo_ranking.sort_values("Điểm LSI", ascending=False, kind="stable"))
display(Markdown("**Bản tóm tắt:** " + demo_summary))

,Câu,Điểm LSI,Được chọn
S6,Bóng đá cần đội bóng mạnh.,0.9134,True
S5,Đội bóng thi đấu bóng đá.,0.9134,True
S2,Ngôn ngữ giúp máy tính hiểu văn bản.,0.7958,False
S3,LSI giúp tóm tắt văn bản.,0.7252,False
S4,Tóm tắt chọn câu trong văn bản.,0.6202,False
S1,Máy tính xử lý ngôn ngữ.,0.5238,False


**Bản tóm tắt:** Đội bóng thi đấu bóng đá. Bóng đá cần đội bóng mạnh.

# Phần 2 - Ứng dụng tóm tắt văn bản trích xuất

Luồng xử lý: **văn bản → tách câu → TF-IDF → SVD → điểm LSI → chọn câu**.
Hàm dưới đây ghép các bước đã mô phỏng. Đầu ra gồm bản tóm tắt và bảng
điểm để giải thích vì sao từng câu được chọn.

In [9]:
def summarize_lsi(text, k=2, sentence_count=3):
    sentences = split_sentences(text)
    tfidf = build_tfidf(sentences)
    # ponytail: SVD trên ma trận đặc, dùng truncated SVD khi xử lý văn bản rất dài.
    u, singular_values, vt = np.linalg.svd(tfidf["matrix"], full_matrices=False)
    scores = semantic_scores(singular_values, vt, k)
    selected_indices, summary = select_sentences(sentences, scores, sentence_count)
    details = pd.DataFrame({
        "Câu": sentences,
        "Điểm LSI": scores,
        "Được chọn": np.isin(np.arange(len(sentences)), selected_indices),
    }, index=[f"S{index + 1}" for index in range(len(sentences))])
    return summary, details

## Sử dụng với văn bản của bạn

Thay nội dung `input_text`, `k` và `sentence_count` rồi chạy lại cell.
Để đọc tệp UTF-8, thay dòng gán bằng `input_text = Path("van_ban.txt").read_text(encoding="utf-8")`.
Chọn `k` và `sentence_count` là số nguyên dương; số câu đầu ra không vượt số
câu đầu vào. Khi giữ toàn bộ chiều SVD, các câu có thể gần bằng điểm do L2.

In [10]:
input_text = demo_text
k = 2
sentence_count = 3

summary, sentence_details = summarize_lsi(input_text, k, sentence_count)
display(sentence_details)
display(Markdown("**Tóm tắt trích xuất:** " + summary))

,Câu,Điểm LSI,Được chọn
S1,Máy tính xử lý ngôn ngữ.,0.5238,False
S2,Ngôn ngữ giúp máy tính hiểu văn bản.,0.7958,True
S3,LSI giúp tóm tắt văn bản.,0.7252,False
S4,Tóm tắt chọn câu trong văn bản.,0.6202,False
S5,Đội bóng thi đấu bóng đá.,0.9134,True
S6,Bóng đá cần đội bóng mạnh.,0.9134,True


**Tóm tắt trích xuất:** Ngôn ngữ giúp máy tính hiểu văn bản. Đội bóng thi đấu bóng đá. Bóng đá cần đội bóng mạnh.

# Phần 3 - Thử nghiệm trên dữ liệu công khai và đánh giá

## 3.1. Dataset CNN/DailyMail

Nguồn: [abisee/cnn_dailymail trên Hugging Face](https://huggingface.co/datasets/abisee/cnn_dailymail).
Sử dụng **50 mẫu đầu tiên của tập test, cấu hình 3.0.0**, tải ngày 07/10/2026.
`article` là bài báo tiếng Anh, `highlights` là tóm tắt tham chiếu,
`id` định danh bài báo. Dữ liệu được giữ nguyên từ nguồn, không tự tạo tham chiếu.

Đây là tập con nhỏ để chạy bài tập; không đại diện cho toàn bộ dataset.
Dữ liệu được lấy qua [Dataset Viewer API](https://huggingface.co/docs/dataset-viewer/rows).
Cell dưới đọc trực tiếp tệp cục bộ đã kèm theo notebook.

In [11]:
dataset_path = Path("data/cnn_dailymail_test_50.json")
records = json.loads(dataset_path.read_text(encoding="utf-8"))

dataset = pd.DataFrame(records)
print("Dataset: CNN/DailyMail | Config: 3.0.0 | Split: test | Số mẫu:", len(dataset))
display(dataset[["id", "article", "highlights"]].head(3))

Dataset: CNN/DailyMail | Config: 3.0.0 | Split: test | Số mẫu: 50


,id,article,highlights
0,f001ec5c4704938247d27a44948eebb37ae98d01,"(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, ...",Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .\nIs...
1,230c522854991d053fe98a718b1defa077a8efef,(CNN)Never mind cats having nine lives. A stray pooch in Washington State has used up at least three of her own afte...,"Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field .\n""She's a true ..."
2,4495ba8f3a340d97a9df1476f8a35502bcce1f69,"(CNN)If you've been following the news lately, there are certain things you doubtless know about Mohammad Javad Zari...",Mohammad Javad Zarif has spent more time with John Kerry than any other foreign minister .\nHe once participated in ...


## 3.2. Tự cài đặt ROUGE-1, ROUGE-2 và ROUGE-L

ROUGE-1/2 đếm unigram/bigram chung, có tính số lần xuất hiện.
ROUGE-L dùng độ dài chuỗi con chung dài nhất (LCS) của chuỗi token.

Với $M$ là số đơn vị chung, $C$ là số đơn vị trong đầu ra và $R$ trong tham chiếu:

$$P=M/C,\quad Recall=M/R,\quad F1=2M/(C+R)$$

Chuỗi không có đơn vị được cho điểm 0. Bộ đo chuyển chữ thường, bỏ dấu câu;
**không stemming, không loại từ dừng**. ROUGE-L ở đây tính trên toàn chuỗi
token, không phải ROUGE-Lsum. Vì khác cấu hình, không so sánh trực tiếp
với điểm từ thư viện dùng stemming hoặc ROUGE-Lsum.

In [12]:
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[index:index + n]) for index in range(len(tokens) - n + 1))


def lcs_length(first, second):
    previous_row = [0] * (len(second) + 1)
    for first_token in first:
        current_row = [0]
        for column, second_token in enumerate(second, 1):
            current_row.append(max(
                previous_row[column],
                current_row[column - 1],
                previous_row[column - 1] + int(first_token == second_token),
            ))
        previous_row = current_row
    return previous_row[-1]


def rouge_scores(reference, prediction):
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    scores = {}
    for n in (1, 2):
        reference_ngrams = ngram_counts(reference_tokens, n)
        prediction_ngrams = ngram_counts(prediction_tokens, n)
        overlap = sum((reference_ngrams & prediction_ngrams).values())
        total = sum(reference_ngrams.values()) + sum(prediction_ngrams.values())
        scores[f"ROUGE-{n}"] = 2 * overlap / max(total, 1)

    common_length = lcs_length(reference_tokens, prediction_tokens)
    total_length = len(reference_tokens) + len(prediction_tokens)
    scores["ROUGE-L"] = 2 * common_length / max(total_length, 1)
    return scores

## 3.3. Thiết lập thí nghiệm

Mỗi phương pháp chọn tối đa **3 câu** trên cùng văn bản và cùng bước tiền xử lý:

- **Lead-3:** lấy ba câu đầu, làm mốc so sánh đơn giản.
- **TF-IDF:** chọn câu theo tổng trọng số TF-IDF sau chuẩn hóa L2.
- **LSI:** chọn câu theo tọa độ ngữ nghĩa, lần lượt `k = 1, 2, 3`.

TF-IDF và SVD được tính riêng cho từng bài báo; không dùng tóm tắt tham chiếu
để chọn câu. Tham chiếu chỉ dùng khi tính ROUGE. Mỗi bài báo chỉ phân rã
SVD một lần và dùng lại cho các giá trị `k`.

In [13]:
experiment_sentence_count = 3
experiment_k_values = [1, 2, 3]
evaluation_rows = []
summary_rows = []

for record in records:
    sentences = split_sentences(record["article"])
    tfidf = build_tfidf(sentences)
    u, singular_values, vt = np.linalg.svd(tfidf["matrix"], full_matrices=False)

    tfidf_indices, tfidf_summary = select_sentences(
        sentences, tfidf["matrix"].sum(axis=0), experiment_sentence_count
    )
    predictions = {
        "Lead-3": " ".join(sentences[:experiment_sentence_count]),
        "TF-IDF": tfidf_summary,
    }
    for k in experiment_k_values:
        scores = semantic_scores(singular_values, vt, k)
        selected_indices, prediction = select_sentences(sentences, scores, experiment_sentence_count)
        predictions[f"LSI (k={k})"] = prediction

    for method, prediction in predictions.items():
        metrics = rouge_scores(record["highlights"], prediction)
        evaluation_rows.append({"id": record["id"], "Phương pháp": method, **metrics})
        summary_rows.append({"id": record["id"], "Phương pháp": method,
                             "Tham chiếu": record["highlights"], "Tóm tắt": prediction})

evaluation = pd.DataFrame(evaluation_rows)
generated_summaries = pd.DataFrame(summary_rows)
average_scores = evaluation.groupby("Phương pháp", sort=False)[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean()
display(Markdown("**Trung bình ROUGE F1 trên 50 bài báo (thang 0–1)**"))
display(average_scores)

**Trung bình ROUGE F1 trên 50 bài báo (thang 0–1)**

,ROUGE-1,ROUGE-2,ROUGE-L
Phương pháp,,,
Lead-3,0.2884,0.1239,0.2132
TF-IDF,0.2126,0.0654,0.1432
LSI (k=1),0.2588,0.0944,0.1857
LSI (k=2),0.2371,0.0771,0.1677
LSI (k=3),0.2416,0.0728,0.1701


## 3.4. Quan sát một ví dụ

Cell này đặt tóm tắt tham chiếu cạnh đầu ra của từng phương pháp và điểm
ROUGE tương ứng. Thay `example_index` để xem các bài báo khác.

In [14]:
example_index = 0
example_id = records[example_index]["id"]
display(Markdown("**Bài báo đầu vào**"))
print(records[example_index]["article"])
display(Markdown("**Tóm tắt tham chiếu**"))
print(records[example_index]["highlights"])
example_summaries = generated_summaries.loc[generated_summaries["id"].eq(example_id)]
for row in example_summaries.to_dict("records"):
    display(Markdown("**" + row["Phương pháp"] + "**"))
    print(row["Tóm tắt"])
display(evaluation.loc[evaluation["id"].eq(example_id)].drop(columns="id"))

**Bài báo đầu vào**

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday's ceremony, sa

**Tóm tắt tham chiếu**

Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


**Lead-3**

...

**LSI (k=2)**

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. "As Palestine formally becomes a State Party to the Rome Statute today, the world is also a step closer to ending a long era of impunity and injustice," he said, according to an ICC news release. "As the Rome Statute today enters into force for the State of Palestine, Palestine acquires all the rights as well as responsibilities that come with being a State Party to the Statute.


**LSI (k=3)**

"As Palestine formally becomes a State Party to the Rome Statute today, the world is also a step closer to ending a long era of impunity and injustice," he said, according to an ICC news release. "As the Rome Statute today enters into force for the State of Palestine, Palestine acquires all the rights as well as responsibilities that come with being a State Party to the Statute. The inquiry will include alleged war crimes committed since June.


,Phương pháp,ROUGE-1,ROUGE-2,ROUGE-L
0,Lead-3,0.2957,0.1593,0.2261
1,TF-IDF,0.2361,0.0845,0.2083
2,LSI (k=1),0.2879,0.1385,0.2273
3,LSI (k=2),0.2290,0.0930,0.2137
4,LSI (k=3),0.2321,0.0364,0.1429


## 3.5. Nhận xét kết quả

Cell dưới đây đọc kết quả thực tế của lần chạy, thay vì ghi sẵn kết luận
LSI luôn tốt hơn. Xếp hạng bằng ROUGE không đánh giá đầy đủ tính mạch lạc,
độ đúng của thông tin hay mức lặp ý.

In [15]:
best_method = average_scores["ROUGE-L"].idxmax()
lsi_scores = average_scores.loc[[f"LSI (k={k})" for k in experiment_k_values]]
best_lsi = lsi_scores["ROUGE-L"].idxmax()
lead_difference = average_scores.loc[best_lsi, "ROUGE-L"] - average_scores.loc["Lead-3", "ROUGE-L"]

print("Phương pháp đạt ROUGE-L cao nhất trên tập mẫu:", best_method)
print("Cấu hình LSI đạt ROUGE-L cao nhất trên tập mẫu:", best_lsi)
print(f"Chênh lệch ROUGE-L của cấu hình LSI đó so với Lead-3: {lead_difference:+.4f}")
print("Chọn thêm chiều ngữ nghĩa không đảm bảo tăng chất lượng; đối chiếu bảng k=1,2,3.")

Phương pháp đạt ROUGE-L cao nhất trên tập mẫu: Lead-3
Cấu hình LSI đạt ROUGE-L cao nhất trên tập mẫu: LSI (k=1)
Chênh lệch ROUGE-L của cấu hình LSI đó so với Lead-3: -0.0275
Chọn thêm chiều ngữ nghĩa không đảm bảo tăng chất lượng; đối chiếu bảng k=1,2,3.


### Kết luận và giới hạn

- Đã mô phỏng đầy đủ TF-IDF → SVD → không gian ngữ nghĩa → chọn câu.
- Đã áp dụng cùng thuật toán để tóm tắt văn bản đầu vào.
- Đã thử nghiệm trên 50 bài báo CNN/DailyMail và đánh giá bằng ba chỉ số ROUGE.
- So sánh các `k` trên cùng tập test nhỏ chỉ mang tính khảo sát, chưa có tập
  validation riêng để chọn tham số; không khẳng định cấu hình tốt nhất cho toàn bộ dataset.
- Tách câu bằng biểu thức chính quy còn hạn chế với chữ viết tắt và số thập phân.
  Tiếng Việt chưa được ghép từ nhiều âm tiết; thử nghiệm chính dùng tiếng Anh.
- Phương pháp chọn câu theo điểm có thể bỏ thông tin quan trọng hoặc chọn các
  câu trùng ý. Với nhiều câu đầu ra, cần đọc ví dụ để đánh giá mức bao phủ và lặp ý.
- ROUGE đo mức trùng token với tham chiếu. Diễn đạt khác nhau vẫn có thể cùng ý,
  nên ROUGE thấp không tự động đồng nghĩa nội dung sai.